# 🧩 Mini-Lab: Baseline Spot-Check

**Module: Ship a Working AI Feature Today** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** why "looks good" is not a valid way to evaluate AI output
2. **Compare** AI outputs against a concrete baseline to judge quality
3. **Identify** where an AI pipeline's output disagrees with a known-good answer

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Baseline Comparison | Judge output against a simple reference baseline (a cheap, clear alternative), not against "looks good". If you cannot beat the baseline, you have not earned complexity. |


## Why You Need a Baseline

When you run your AI pipeline and see the output, the natural reaction is:

> *"That looks right to me."* ✅ Ship it!

This is **dangerous**. "Looks good" means:
- You're judging quality with your gut, not with data
- You can't tell if a code change made things *worse*
- You have no way to communicate quality to your team

The fix is simple: **create a small set of examples where YOU write the correct answer first**, then compare the AI's output against your answers. That's a **baseline**.

```
┌──────────────┐      ┌──────────────┐
│  AI Output   │ vs.  │   Baseline   │  →  Match? Yes/No
│ (predicted)  │      │ (your label) │
└──────────────┘      └──────────────┘
```

In [1]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## Step 1: Create the Baseline

Let's reuse the support-ticket triage scenario from `lab-triage-csv-rows`. We have five tickets, and **we** label the correct category for each one. This is the baseline — the source of truth.

In [2]:
# The baseline: YOU decide the correct answer for each ticket
baseline = [
    {"ticket_id": "101", "message": "I was charged twice for my subscription last month.",         "expected": "Billing"},
    {"ticket_id": "102", "message": "The app crashes every time I try to upload a file.",         "expected": "Technical"},
    {"ticket_id": "103", "message": "How do I change the email address on my account?",           "expected": "Account"},
    {"ticket_id": "104", "message": "I love your product! Just wanted to say thanks.",            "expected": "General"},
    {"ticket_id": "105", "message": "My invoice shows a charge for a plan I never signed up for.", "expected": "Billing"},
]

print(f"Baseline: {len(baseline)} tickets with human-labelled categories")

Baseline: 5 tickets with human-labelled categories


## Step 2: Simulate AI Predictions

In a real pipeline you'd call the LLM (as you did in `lab-triage-csv-rows`). Here we simulate the AI's output so we can focus purely on the **comparison** skill — no API key needed.

In [3]:
# Simulated AI predictions (one is deliberately wrong)
ai_predictions = {
    "101": "Billing",
    "102": "Technical",
    "103": "General",   # ← Wrong! Should be "Account"
    "104": "General",
    "105": "Billing",
}

print("AI predictions loaded (5 tickets)")

AI predictions loaded (5 tickets)


## Step 3: Compare AI Output Against the Baseline

This is the core skill: walk through each example, compare predicted vs. expected, and flag mismatches.

In [4]:
def spot_check(baseline, predictions):
    """Compare AI predictions against a human-labelled baseline."""
    results = []
    for item in baseline:
        tid = item["ticket_id"]
        expected = item["expected"]
        predicted = predictions.get(tid, "MISSING")
        match = predicted == expected
        results.append({
            "ticket_id": tid,
            "message": item["message"][:50],
            "expected": expected,
            "predicted": predicted,
            "match": match,
        })
    return results

results = spot_check(baseline, ai_predictions)

# Build a results table
header = "| Ticket | Message | Expected | Predicted | Match |\n|--------|---------|----------|-----------|-------|\n"
rows = ""
for r in results:
    icon = "✅" if r["match"] else "❌"
    rows += f"| {r['ticket_id']} | {r['message']}… | {r['expected']} | {r['predicted']} | {icon} |\n"

correct = sum(1 for r in results if r["match"])
total = len(results)
score = {"correct": correct, "total": total}

md(f"## Spot-Check Results\n\n{header}{rows}\n**Score: {correct}/{total}**")

## Spot-Check Results

| Ticket | Message | Expected | Predicted | Match |
|--------|---------|----------|-----------|-------|
| 101 | I was charged twice for my subscription last month… | Billing | Billing | ✅ |
| 102 | The app crashes every time I try to upload a file.… | Technical | Technical | ✅ |
| 103 | How do I change the email address on my account?… | Account | General | ❌ |
| 104 | I love your product! Just wanted to say thanks.… | General | General | ✅ |
| 105 | My invoice shows a charge for a plan I never signe… | Billing | Billing | ✅ |

**Score: 4/5**

## Step 4: Inspect the Mismatches

A score of 4/5 tells you *how well* the pipeline did. But the real value is looking at **which ones it got wrong and why**. That's what turns a number into an actionable fix.

In [5]:
mismatches = [r for r in results if not r["match"]]

if mismatches:
    output = "## ❌ Mismatches to Investigate\n\n"
    for m in mismatches:
        output += (
            f"**Ticket {m['ticket_id']}:** \"{m['message']}…\"\n\n"
            f"- Expected: **{m['expected']}**\n"
            f"- Predicted: **{m['predicted']}**\n"
            f"- 💡 *Ask: Is the prompt clear enough to distinguish {m['expected']} from {m['predicted']}?*\n\n---\n\n"
        )
    md(output)
else:
    md("## ✅ No Mismatches\n\nAll predictions match the baseline.")

## ❌ Mismatches to Investigate

**Ticket 103:** "How do I change the email address on my account?…"

- Expected: **Account**
- Predicted: **General**
- 💡 *Ask: Is the prompt clear enough to distinguish Account from General?*

---



## "Looks Good" vs. Baseline: The Difference

Without the baseline, ticket 103 would have slipped through — "General" *sounds* reasonable for an email-change request. The baseline caught it because **you decided the answer ahead of time**, before seeing the AI's output.

| Approach | What You Know | Risk |
|----------|---------------|------|
| "Looks good" | The output *seems* fine to you right now | Subtle errors slip through; no way to track quality over time |
| **Baseline spot-check** | Exactly which examples the AI got right and wrong | You know your accuracy *before* shipping and can improve it |

## 🎯 Summary

### Key Takeaways

1. **Baseline first, AI second** — label a small set of examples with the correct answers *before* you look at the AI's output; this prevents your judgment from being biased by what the model says
2. **Compare, don't eyeball** — a simple match/mismatch check on 5 examples is more reliable than reading 50 outputs and deciding they "look good"
3. **Mismatches are actionable** — each wrong answer points to a specific fix (better prompt wording, clearer category definitions, or a pipeline bug)